# Phase 0: OpenAI Agents SDK + Codex Tool Spike

Test that the architecture works before committing to migration.

This notebook validates:
1. LitellmModel with MiniMax (default) and ZAI providers
2. Custom `write_memory` FunctionTool
3. Agent + write_memory integration
4. Codex tool for filesystem exploration (via local ResponsesProxy)
5. Combined architecture (Codex + write_memory)
6. Sandbox isolation

In [ ]:
"""Phase 0: OpenAI Agents SDK + Codex Tool Spike
Test that the architecture works before committing to migration.
"""
import os
import sys
import json
from pathlib import Path
from dataclasses import dataclass
from datetime import datetime, timezone

# Load .env
from dotenv import load_dotenv
load_dotenv(
	Path(__file__).parent.parent / ".env"
	if "__file__" in dir()
	else Path("/Users/kargarisaac/codes/personal/lerim/lerim-cli/.env")
)

# Verify keys
for key in ["MINIMAX_API_KEY", "ZAI_API_KEY", "OPENROUTER_API_KEY", "OPENAI_API_KEY"]:
	val = os.environ.get(key, "")
	print(f"{key}: {'set (' + val[:8] + '...)' if val else 'NOT SET'}")

## Test 1: LitellmModel with Multiple Providers

Verify that we can use non-OpenAI models (MiniMax, ZAI) through the `LitellmModel` extension.
This is critical because Lerim must support arbitrary model providers.

In [ ]:
"""Test 1: Can we use non-OpenAI models via LitellmModel?"""
from agents import Agent, Runner, function_tool, set_tracing_disabled
from agents.extensions.models.litellm_model import LitellmModel

set_tracing_disabled(disabled=True)

# --- Provider configs ---
# MiniMax M2.5 (default for lerim)
minimax_model = LitellmModel(
	model="minimax/MiniMax-M2.5",
	api_key=os.environ["MINIMAX_API_KEY"],
)

# ZAI / GLM-4.7 (fallback) — uses OpenAI-compatible API with custom base_url
zai_model = LitellmModel(
	model="openai/glm-4.7",
	api_key=os.environ["ZAI_API_KEY"],
	base_url="https://open.bigmodel.cn/api/paas/v4",
)

# Default model for all subsequent tests
model = minimax_model

# --- Test each provider ---
for name, m in [("MiniMax", minimax_model), ("ZAI/GLM-4.7", zai_model)]:
	try:
		agent = Agent(
			name="TestAgent",
			instructions="You are a helpful assistant. Respond concisely in 1-2 sentences.",
			model=m,
		)
		result = await Runner.run(agent, "What is the capital of France?")
		print(f"{name}: {result.final_output}")
	except Exception as e:
		err = str(e)
		if "429" in err or "余额" in err:
			print(f"{name}: Connected OK but account has insufficient balance (expected)")
		else:
			print(f"{name}: FAILED - {err[:200]}")

print("\nMulti-provider test complete!")

## Test 2: Custom FunctionTool (write_memory)

Verify that we can define a `write_memory` tool using the `@function_tool` decorator.
This replaces PydanticAI's tool definition pattern. The tool validates inputs and writes
structured markdown memory files.

In [ ]:
"""Test 2: Can we create a custom write_memory tool that works alongside agents?"""
from agents import function_tool, RunContextWrapper
import tempfile


@dataclass
class LerimContext:
	memory_root: Path
	run_id: str


def _slugify(text: str) -> str:
	import re
	return re.sub(r'[^a-z0-9]+', '-', text.lower()).strip('-')


@function_tool
def write_memory(
	ctx: RunContextWrapper[LerimContext],
	primitive: str,
	title: str,
	body: str,
	confidence: float = 0.8,
	tags: str = "",
	kind: str = "",
) -> str:
	"""Write a structured memory record to the memory folder.

	Args:
		primitive: Must be 'decision' or 'learning'
		title: Short descriptive title
		body: Memory content
		confidence: Confidence score 0.0-1.0
		tags: Comma-separated tags
		kind: Required for learnings: insight, procedure, friction, pitfall, or preference
	"""
	# Validation (replaces PydanticAI's ModelRetry)
	if primitive not in ("decision", "learning"):
		return f"ERROR: primitive must be 'decision' or 'learning', got '{primitive}'"
	if not title or not title.strip():
		return "ERROR: title cannot be empty"
	if not (0.0 <= confidence <= 1.0):
		return f"ERROR: confidence must be 0.0-1.0, got {confidence}"
	valid_kinds = {"insight", "procedure", "friction", "pitfall", "preference"}
	if primitive == "learning" and kind not in valid_kinds:
		return f"ERROR: learning requires kind in {valid_kinds}, got '{kind}'"

	# Build markdown
	now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
	tag_list = [t.strip() for t in tags.split(",") if t.strip()] if tags else []

	lines = [
		"---",
		f"id: {_slugify(title)}",
		f'title: "{title}"',
		f"confidence: {confidence}",
		f"tags: {json.dumps(tag_list)}",
	]
	if kind:
		lines.append(f"kind: {kind}")
	lines.extend([
		f"source: {ctx.context.run_id}",
		f"created: {now}",
		f"updated: {now}",
		"---",
		"",
		body,
		"",
	])
	content = "\n".join(lines)

	# Write file
	folder = "decisions" if primitive == "decision" else "learnings"
	slug = _slugify(title)
	filename = f"{now[:10].replace('-', '')}-{slug}.md"
	target = ctx.context.memory_root / folder / filename
	target.parent.mkdir(parents=True, exist_ok=True)
	target.write_text(content, encoding="utf-8")

	return json.dumps({"file_path": str(target), "bytes": len(content.encode()), "primitive": primitive})


# Test write_memory directly
with tempfile.TemporaryDirectory() as tmp:
	ctx_obj = LerimContext(memory_root=Path(tmp), run_id="spike-test")

	# Create a mock RunContextWrapper (for direct testing)
	print("write_memory tool created successfully")
	print(f"Tool name: {write_memory.name}")
	print(f"Tool params: {json.dumps(write_memory.params_json_schema, indent=2)}")
	print("Custom FunctionTool works!")

## Test 3: Agent with write_memory tool

Verify that a LitellmModel-backed agent can use the `write_memory` tool to create
structured memory files. This is the core write path for the new architecture.

In [ ]:
"""Test 3: Agent uses write_memory tool with LitellmModel"""

memory_agent = Agent(
	name="MemoryWriter",
	instructions="""You are a memory extraction agent. When asked to save a memory,
use the write_memory tool. Always provide primitive, title, body, and appropriate fields.
For learnings, always include kind (insight, procedure, friction, pitfall, or preference).""",
	model=model,
	tools=[write_memory],
)

with tempfile.TemporaryDirectory() as tmp:
	ctx = LerimContext(memory_root=Path(tmp), run_id="spike-test")

	result = await Runner.run(
		memory_agent,
		"Save a decision: 'Use PostgreSQL for all persistence' with confidence 0.9 and tags 'database,infrastructure'",
		context=ctx,
	)

	print(f"Agent response: {result.final_output}")

	# Check if file was written
	decisions = list(Path(tmp).glob("decisions/*.md"))
	if decisions:
		print(f"\nWritten file: {decisions[0].name}")
		print(decisions[0].read_text())
		print("Agent + write_memory tool works!")
	else:
		print("No memory file written in decisions/")
		learnings = list(Path(tmp).glob("learnings/*.md"))
		if learnings:
			print(f"Found in learnings: {learnings[0].name}")
			print(learnings[0].read_text())

## Test 4: Codex Tool (filesystem sub-agent)

Test the Codex tool for filesystem exploration. Uses a local ResponsesProxy
to translate Responses API → Chat Completions for MiniMax.

In [ ]:
"""Test 4: Codex tool for filesystem operations
Uses local ResponsesProxy to bridge MiniMax (Chat Completions) for Codex CLI (Responses API).
"""
import sys
sys.path.insert(0, str(Path("/Users/kargarisaac/codes/personal/lerim/lerim-cli/.claude/worktrees/agent-af7d097c/src")))

from agents.extensions.experimental.codex import codex_tool, ThreadOptions, TurnOptions, CodexOptions
from lerim.runtime.responses_proxy import ResponsesProxy

# Create a test directory with some memory files
test_dir = Path(tempfile.mkdtemp(prefix="lerim-spike-"))
(test_dir / "decisions").mkdir()
(test_dir / "learnings").mkdir()

for i, (title, body, conf) in enumerate([
	("Use PostgreSQL", "All persistence should use PostgreSQL for consistency.", 0.95),
	("Use tabs for indentation", "Project convention: tabs not spaces in all code.", 0.9),
	("JWT with 24h expiry", "Authentication tokens expire after 24 hours.", 0.85),
]):
	slug = _slugify(title)
	content = f"""---
id: {slug}
title: "{title}"
confidence: {conf}
tags: ["test"]
source: spike-seed
created: 2026-03-25T00:00:00Z
updated: 2026-03-25T00:00:00Z
---

{body}
"""
	(test_dir / "decisions" / f"2026032{i}-{slug}.md").write_text(content)

print(f"Test directory: {test_dir}")
print(f"Test memories: {list((test_dir / 'decisions').glob('*.md'))}")

# Start local proxy: Responses API → MiniMax Chat Completions
proxy = ResponsesProxy(
	backend_url="https://api.minimax.io/v1",
	api_key=os.environ["MINIMAX_API_KEY"],
)
proxy.start()
print(f"Proxy running at {proxy.url}")

codex_agent = Agent(
	name="CodexExplorer",
	instructions=f"""You are a memory exploration agent. Use the codex tool to explore
the memory directory and answer questions about existing memories.
The memory directory is at: {test_dir}
Look in the decisions/ and learnings/ subdirectories for .md files.""",
	model=model,
	tools=[
		codex_tool(
			codex_options=CodexOptions(
				base_url=proxy.url,
				api_key="proxy-managed",
			),
			sandbox_mode="read-only",
			working_directory=str(test_dir),
			skip_git_repo_check=True,
			default_thread_options=ThreadOptions(
				model="MiniMax-M2.5",
				approval_policy="never",
				network_access_enabled=False,
			),
			default_turn_options=TurnOptions(
				idle_timeout_seconds=60,
			),
		),
	],
)

try:
	result = await Runner.run(
		codex_agent,
		"List all memory files in the decisions/ directory and tell me what decisions exist. Include their confidence scores.",
	)
	print(f"\nAgent response:\n{result.final_output}")
	print(f"\nCodex tool with MiniMax via local proxy works!")
finally:
	proxy.stop()
	print("Proxy stopped")

## Test 5: Combined Agent (Codex + write_memory)

Full architecture test: Codex reads existing memories via local proxy,
write_memory creates new structured memories. Both use MiniMax.

In [ ]:
"""Test 5: Full architecture -- Codex + write_memory, both using MiniMax"""

proxy = ResponsesProxy(
	backend_url="https://api.minimax.io/v1",
	api_key=os.environ["MINIMAX_API_KEY"],
)
proxy.start()

full_agent = Agent(
	name="LerimSyncAgent",
	instructions=f"""You are a memory management agent. You have two capabilities:
1. Use the codex tool to explore and read files in the memory directory at: {test_dir}
2. Use the write_memory tool to create new structured memories

Your task:
- First, use codex to read existing memories in decisions/
- Then, create a NEW learning memory that doesn't duplicate existing ones
- The learning should be about something useful for the project""",
	model=model,
	tools=[
		codex_tool(
			codex_options=CodexOptions(
				base_url=proxy.url,
				api_key="proxy-managed",
			),
			sandbox_mode="workspace-write",
			working_directory=str(test_dir),
			skip_git_repo_check=True,
			default_thread_options=ThreadOptions(
				model="MiniMax-M2.5",
				approval_policy="never",
				network_access_enabled=False,
			),
			default_turn_options=TurnOptions(
				idle_timeout_seconds=60,
			),
		),
		write_memory,
	],
)

ctx = LerimContext(memory_root=test_dir, run_id="spike-combined")

try:
	result = await Runner.run(
		full_agent,
		"Read the existing decisions, then create a new learning (kind=insight) about database connection pooling best practices.",
		context=ctx,
	)

	print(f"Agent response:\n{result.final_output}")

	all_files = list(test_dir.rglob("*.md"))
	print(f"\nAll memory files ({len(all_files)}):")
	for f in sorted(all_files):
		print(f"  {f.relative_to(test_dir)}")

	print("\nFull architecture (Codex + write_memory) works!")
finally:
	proxy.stop()
	print("Proxy stopped")

## Test 6: Sandbox Isolation

Verify that the Codex sandbox prevents the agent from accessing files outside
the designated working directory.

In [ ]:
"""Test 6: Verify Codex sandbox prevents access outside working directory"""

proxy = ResponsesProxy(
	backend_url="https://api.minimax.io/v1",
	api_key=os.environ["MINIMAX_API_KEY"],
)
proxy.start()

sandbox_agent = Agent(
	name="SandboxTest",
	instructions="Try to read the file /etc/passwd using the codex tool. Report what happens.",
	model=model,
	tools=[
		codex_tool(
			codex_options=CodexOptions(
				base_url=proxy.url,
				api_key="proxy-managed",
			),
			sandbox_mode="read-only",
			working_directory=str(test_dir),
			skip_git_repo_check=True,
			default_thread_options=ThreadOptions(
				model="MiniMax-M2.5",
				approval_policy="never",
				network_access_enabled=False,
			),
		),
	],
)

try:
	result = await Runner.run(sandbox_agent, "Read the file /etc/passwd and show its contents.")
	print(f"Sandbox test response:\n{result.final_output}")
	print("\n(If the agent could NOT read /etc/passwd, sandbox is working)")
finally:
	proxy.stop()
	print("Proxy stopped")

## Cleanup

In [ ]:
"""Cleanup test directories"""
import shutil
if test_dir.exists():
	shutil.rmtree(test_dir)
	print(f"Cleaned up {test_dir}")
print("\nPhase 0 Spike Complete!")
print("Summary:")
print("  - LitellmModel with MiniMax and ZAI")
print("  - Custom write_memory FunctionTool")
print("  - Agent + write_memory integration")
print("  - Codex tool via local ResponsesProxy (no OpenRouter)")
print("  - Combined architecture (Codex + write_memory)")
print("  - Sandbox isolation")